# Recommender systems

## Introduction

In this bonus chapter we quickly discuss recommender systems.

This chapter is based on this [excellent lecture](https://mlvu.github.io/lecture12/).
We cover much less, but we show how to implement some things using `pytroch`.

## Introduction

We are going to make a movie recommendation system, so we will use terminology related to this problem.

However, there are many different applications, for example any social media feed utilizes some sort of recommender system.

## Problem Description

We are given some ratings of several movies by users.
The goal is to predict what rating a specific user would give to a movie if they rated it.

## Problem Description

So a mathematical description of the problem could be as follows.

We are given two sets $U$ and $M$ and a function $r: U \times M \rightarrow \mathbb{R}$.
Also we know what value $r$ takes at some points $(u, m),$ but not all of them.
The goal is to extrapolate the rest of function values.

## Problem Description

So in our case $U$ is the set of users, $M$ is the set of movies and $r(u, v)$ is what rating user $u$ gives to movie $m$.
We will use the function $r$ to denote the rating throughout the chapter.

## Matrix Decomposition Approach

Here is one way to conceptualize this problem.

We may suppose that there are $N$ features (that is $N$ numbers) that describe any possible movie.
For example, how actiony or romantic the movie is.
More mathematical way to express this is that there is an embedding $M \rightarrow \mathbb{R}^N$.

Also, we can suppose each user's taste in movies is completely determined by how much they like or dislike each of these $N$ features.
And also we can assume that this like / dislike can be expressed by $N$ numbers.
So we have another embedding $U \rightarrow \mathbb{R}^N$.

## Matrix Decomposition Approach

Denote the $i$-th user's embedding by $u_i$ and $j$-th movie's embedding by $m_j$.
We can then try to compute the rating given using the scalar product:
$$
  r(u_i, m_j) = \langle u_i, m_j \rangle.
$$

## Matrix Decomposition Approach

For example, if the movie is not romantic and the user dislikes romantic movies then the hypothetical feature corresponding to how romantic the movie would be negative in both $u_i$ and $m_j$ so their product would be positive and the rating would increase, as expected.

Just to clarify, we do not know that one of the $N$ features measures how romantic the movie is, this is just an example to make the intuition clearer.

## Matrix Decomposition Approach

If we imagine $U$ and $M$ to be the matrices where rows are embedded users and movies respectively then the full rating cross table $R$ is given by
$$
  R = UM^T.
$$
I.e. $R$ is the matrix where the $(i, j)$ entry is the rating given by the $i$-th user to the $j$-th movie.

You should now be able to guess why this is called the matrix decomposition approach.

## Matrix Decomposition Approach

Now, of course we have no idea what these $N$ features could be or how we would go about measuring them in real life.

The only thing we know is some entries of $R$.

The main idea is to let the computer learn these $N$ features by starting with random embeddings and optimizing using known entries of $R$.

## Matrix Decomposition Approach

For example, we can do this using one of the variants of gradient descent by optimizing the following loss function
$$
  \frac{1}{n} \sum_{\text{known } (i, j)} (R_{ij} - r(u_i, m_j))^2,
$$
here we sum over the ratings that we know, $R_{ij}$ is the rating that $i$-th user has given to $j$-th movie, $n$ is the number of known ratings.

## Matrix Decomposition Approach

The weights of the model are implicit in the above formula - they are the embeddings that we use for users and movies.

As a reminder, this loss function is called mean squared error (MSE).

## Matrix Decomposition Approach

In real life ratings will be biased - some mouvies are objectively better than others and some users are more critical than others.

To account for this we can add bias terms to our rating function.
We add separate bias terms for each user and movie and also an overall general bias term.
These terms will also be weights of our model.

## Matrix Decomposition Approach

So now our rating function takes the form
$$
  r(u_i, m_j) = \langle u_i, m_j \rangle + \text{user_b}_i + \text{movie_b}_j + \text{general_b}.
$$

## Matrix Decomposition Approach

How do we validate such a model?

First logical guess would be that we withhold some users or movies from our training dataset and see how the model performs on those.
The problem is that if we withhold some users or movies then the model will not learn the embedding for them and will not be able to function.

What we actually need to do is to withhold some ratings from our dataset and see if the model can guess them correctly.

## Matrix Decomposition Approach

Okay, let's check out how to implement this using `pytorch`. We'll use the movie lens dataset. Let's first load it and split off a testing set.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ratings = pd.read_csv('https://raw.githubusercontent.com/jputrius/ml_intro/refs/heads/main/data/movielens/ratings.csv')
movies = pd.read_csv('http://raw.githubusercontent.com/jputrius/ml_intro/refs/heads/main/data/movielens/movies.csv')

movies = movies.reset_index(names="movieIndex")
ratings = ratings.merge(movies[["movieIndex", "movieId"]], on="movieId")
ratings["userIndex"] = ratings["userId"] - 1


TRAIN, TEST = train_test_split(ratings, test_size=0.2, random_state=34)
NO_USERS = ratings["userIndex"].nunique()
NO_MOVIES = movies["movieIndex"].nunique()

## Matrix Decomposition Approach

Now let's build a dataset class.

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader

class Ratings(Dataset):
  def __init__(self, train=False):
    if train:
      self.dataset = TRAIN
    else:
      self.dataset = TEST

  def __len__(self):
    return len(self.dataset)

  def __getitem__(self, idx):
    row = self.dataset.iloc[idx]
    
    return torch.LongTensor([row["userIndex"]]), torch.LongTensor([row["movieIndex"]]), torch.FloatTensor([row["rating"]])

batch_size = 64

train_data = Ratings(
  train=True
)

test_data = Ratings(
  train=False
)

train_dataloader = DataLoader(
  train_data,
  batch_size=batch_size,
  shuffle=True
)

test_dataloader = DataLoader(
  test_data,
  batch_size=batch_size,
  shuffle=False
)

## Matrix Decomposition Approach

Now let's build the model itself. It's just a bunch of `nn.Embedding` layers.

In [3]:
import torch
from torch import nn

class Recommender(nn.Module):
  def __init__(self, no_users, no_movies, embed_dim):
    super().__init__()
    self.embed_dim = embed_dim
    self.no_users = no_users
    self.no_movies = no_movies

    self.user_embedding = nn.Embedding(embedding_dim=embed_dim, num_embeddings=no_users)
    self.movie_embedding = nn.Embedding(embedding_dim=embed_dim, num_embeddings=no_movies)
    self.user_bias_embedding = nn.Embedding(embedding_dim=1, num_embeddings=no_users)
    self.movie_bias_embedding = nn.Embedding(embedding_dim=1, num_embeddings=no_movies)
    self.general_bias = nn.Parameter(torch.randn(1))

  def forward(self, u, m):
    users = self.user_embedding(u).squeeze()
    user_bias = self.user_bias_embedding(u).squeeze()

    movies = self.movie_embedding(m).squeeze()
    movie_bias = self.movie_bias_embedding(m).squeeze()

    scores = torch.linalg.vecdot(users, movies) + user_bias + movie_bias + self.general_bias
    
    return scores.unsqueeze(1)

  def get_movie_embeddings(self, m):
    return self.movie_embedding(m).squeeze().cpu()

## Matrix Decomposition Approach

Copy over training code.

In [4]:
from tqdm import tqdm # This is a library that implements loading bars
import sys

def train_epoch(dataloader, model, loss_fn, optimizer):
  model.train() # Set model to training mode

  total_loss = 0
  total_batches = 0

  with tqdm(dataloader, unit="batch", file=sys.stdout) as ep_tqdm:
    ep_tqdm.set_description("Train")
    for U, M, r in ep_tqdm:
      U, M, r = U.to(device), M.to(device), r.to(device)

      # Forward pass
      pred = model(U, M)
      loss = loss_fn(pred, r)
        
      # Backward pass
      loss.backward()
      optimizer.step()

      # Reset the computed gradients back to zero
      optimizer.zero_grad()

      # Output stats
      total_loss += loss
      total_batches += 1
      ep_tqdm.set_postfix(average_batch_loss=(total_loss/total_batches).item())

def eval_epoch(dataloader, model, loss_fn):
  model.eval() # Set model to inference mode
  
  total_loss = 0
  total_batches = 0
  total_samples = 0
  total_correct = 0

  with torch.no_grad(): # Do not compute gradients
    with tqdm(dataloader, unit="batch", file=sys.stdout) as ep_tqdm:
      ep_tqdm.set_description("Val")
      for U, M, r in ep_tqdm:
        U, M, r = U.to(device), M.to(device), r.to(device)
        pred = model(U, M)

        total_loss += loss_fn(pred, r)
        total_batches += 1

        ep_tqdm.set_postfix(average_batch_loss=(total_loss/total_batches).item())

## Matrix Decomposition Approach

And train!

In [5]:
#| output-location: slide
# Hyperparameters
learning_rate = 0.001
epochs = 20

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

model = Recommender(NO_USERS, NO_MOVIES, 16).to(device)

loss_fn = nn.MSELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Organize the training loop
for t in range(epochs):
  print(f"Epoch {t+1}\n-------------------------------")
  train_epoch(train_dataloader, model, loss_fn, optimizer)
  eval_epoch(test_dataloader, model, loss_fn)

print("Done!")

Using cuda device
Epoch 1
-------------------------------
Val: 100%|██████████| 316/316 [00:00<00:00, 364.76batch/s, average_batch_loss=20.7]
Epoch 2
-------------------------------
Val: 100%|██████████| 316/316 [00:00<00:00, 393.11batch/s, average_batch_loss=13.5]
Epoch 3
-------------------------------
Val: 100%|██████████| 316/316 [00:00<00:00, 393.68batch/s, average_batch_loss=9.69]
Epoch 4
-------------------------------
Val: 100%|██████████| 316/316 [00:00<00:00, 383.97batch/s, average_batch_loss=7.51]
Epoch 5
-------------------------------
Val: 100%|██████████| 316/316 [00:00<00:00, 378.40batch/s, average_batch_loss=6.04]
Epoch 6
-------------------------------
Val: 100%|██████████| 316/316 [00:00<00:00, 423.32batch/s, average_batch_loss=4.98]
Epoch 7
-------------------------------
Val: 100%|██████████| 316/316 [00:00<00:00, 378.60batch/s, average_batch_loss=4.2] 
Epoch 8
-------------------------------
Val: 100%|██████████| 316/316 [00:00<00:00, 379.06batch/s, average_batch_l

## Binary Datasets

What to do if our system collect not rating but only likes? For example, in our platform the user gives likes instead of a star rating.

We then only know that a user likes a movie. If a user has watched a movie and has not pressed like then we don't even know if a user dislikes it or is ambivalent about it.

## Binary Datasets

The first problem to overcome here is how to collect negative samples, i.e. how to get samples where a user dislikes a movie.
Interestingly, what is done is that negative samples are picked randomly out of those samples that haven't been liked.
The intuition here is that most movies are crap so if we pick randomly we have a high chance of picking correctly.

So what we do is for each positive sample we randomly pick $n$ non-positive samples to be negative, where $n$ is a hyperparameter.

## Binary Datasets

Then we pivot the model to output a confidence that a user likes a movie instead of a rating. We can do this as follows:
$$
  r(u_i, m_j) = \sigma(\langle u_i, m_j \rangle + \text{user_b}_i + \text{movie_b}_j + \text{general_b}),
$$
where $\sigma$ is, for example, the logistic function.

We then use binary cross-entropy when training instead of MSE.

## Extras

This exposition was based on this [excellent lecture](https://mlvu.github.io/lecture12/).
The lecture covers much more so go ahead and read it (there is also a youtube video)!

## Practice task

In the movie lens dataset define that a user likes a movie if the rating that they gave is $4$ or more.

Write a recommender that trains on this binary data.